# Flux
[Flux](https://flux-framework.readthedocs.io) is a modern HPC resource manager and job scheduler similar to SLURM, but designed for hierarchical scheduling and many-task workflows. 

The basic user interaction is very similar:
| SLURM | Flux | Purpose |
|---|---|---|
| `sbatch` | `flux submit` | submit a batch job |
| `srun` | `flux run` | launch tasks interactively |
| `squeue` | `flux jobs` | inspect jobs |

Read more about flux: 
* [Integrate flux with SLURM](https://flux-framework.readthedocs.io/projects/flux-core/en/stable/guide/start.html#starting-with-slurm)
* [Cheat sheet](https://flux-framework.org/cheat-sheet/)
* [Flux Learning Guide](https://flux-framework.readthedocs.io/en/latest/guides/learning_guide.html)

## Basic Flux Commands
### `flux start`
`flux start` launches a new Flux scheduler instance inside an existing allocation.

This is commonly used together with SLURM:

```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun flux start workflow.sh
```

And in the workflow script `workflow.sh` we start the individual tasks as before: 
```bash
flux run -n 4 python script.py &
flux run -n 4 python script.py &
wait
```

In the jupyter environment the following commands can be directly executed.

### `flux resource list`

Inspect the available computing resources:
```bash
flux resource list
```

Example output:
```
     STATE NNODES NCORES NGPUS NODELIST
      free      1     24     0 jupyter-pyiron-workshop-torlib-tutorial-fiekunpl
 allocated      0      0     0 
      down      0      0     0 
```

In [1]:
!flux resource list

     STATE NNODES NCORES NGPUS NODELIST
      free      1     24     0 jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
 allocated      0      0     0 
      down      0      0     0 


### `flux jobs`

Inspect the Flux queue.
```bash
flux jobs
```
Show all jobs including completed jobs:
```bash
flux jobs -a
```

In [2]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO


### `flux submit`
Submit a non-interactive job to the Flux scheduler.
```bash
flux submit --nodes=1 --ntasks=4 python script.py
```
Returns a Flux job ID.

In [3]:
lines = """\
from mpi4py import MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
print(rank, size)
"""

with open("script.py", "w") as f:
    f.writelines(lines)

In [4]:
!flux submit -o pmi=pmix --nodes=1 --ntasks=4 python script.py

ƒ76Mhphu


In [5]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒ76Mhphu jovyan   python     CD      4      1   1.836s jupyter-pyiron-workshop-torlib-tutorial-bpficiu6


In [6]:
!flux job attach ƒ76Mhphu

1 4
0 4
3 4
2 4


### `flux run`
Launch a blocking interactive job.
```bash
flux run -n 4 python script.py
```
Equivalent in spirit to interactive srun.

In [7]:
!flux run -o pmi=pmix -n 4 python script.py

3 4
1 4
0 4
2 4


In [8]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
    ƒBftPC7y jovyan   python     CD      4      1   1.837s jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
    ƒ76Mhphu jovyan   python     CD      4      1   1.836s jupyter-pyiron-workshop-torlib-tutorial-bpficiu6


## mpi4py Examples with Flux
### Multiple flux run Calls
In analogy to the SLURM example run with:
```bash
flux run -o pmi=pmix -n 4 python script.py &
flux run -o pmi=pmix -n 4 python script.py &
wait
```
This launches two independent MPI jobs of size 4 each.

In [9]:
!flux run -o pmi=pmix -n 4 python script.py & flux run -o pmi=pmix -n 4 python script.py & wait

2 4
1 4
0 4
3 4
2 4
0 4
3 4
1 4


### One flux run and Split the MPI Communicator
Save as `mpisplit.py`:

```python
from mpi4py import MPI

def task(comm):
    rank = comm.Get_rank()
    size = comm.Get_size()
    print(rank, size)

world = MPI.COMM_WORLD
rank = world.Get_rank()
color = 0 if rank < 4 else 1
subcomm = world.Split(color=color, key=rank)
if color == 0:
    # ranks 0–3 run task A
    task(subcomm)
else:
    # ranks 4–7 run task B
    task(subcomm)
```

Run with:
```bash
flux run -o pmi=pmix -n 8 python mpisplit.py
```
This launches one MPI job of size 8 and divides it into two logical communicators of size 4.

In [10]:
mpi_lines = """\
from mpi4py import MPI

def task(comm):
    rank = comm.Get_rank()
    size = comm.Get_size()
    print(rank, size)

world = MPI.COMM_WORLD
rank = world.Get_rank()
color = 0 if rank < 4 else 1
subcomm = world.Split(color=color, key=rank)
if color == 0:
    # ranks 0–3 run task A
    task(subcomm)
else:
    # ranks 4–7 run task B
    task(subcomm)
"""

with open("mpisplit.py", "w") as f:
    f.writelines(mpi_lines)

In [11]:
!flux run -o pmi=pmix -n 8 python mpisplit.py

3 4
2 4
1 4
0 4
3 4
2 4
1 4
0 4


## Flux Python Interface
```python
import flux.job

jobid = flux.job.submit(
    flux.Flux(),
    jobspec=flux.job.JobspecV1.from_command(
        command=["hostname"],
        num_tasks=4,
    ),
)

print(jobid)
```

In [12]:
import flux.job

jobid = flux.job.submit(
    flux.Flux(),
    jobspec=flux.job.JobspecV1.from_command(
        command=["hostname"],
        num_tasks=4,
    ),
)

print(jobid)

ƒMhJMY7R


In [13]:
!flux job attach $jobid

jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
jupyter-pyiron-workshop-torlib-tutorial-bpficiu6
